# 🚀 Time Series — LSTM Training dengan GPU
## PyTorch + Google Colab GPU

> Notebook untuk melatih model **LSTM** pada data deret waktu menggunakan **GPU** di Google Colab.
> Jika dijalankan di laptop tanpa GPU, otomatis fallback ke CPU.
> Dataset tetap dari file CSV agar hasilnya **identik** di semua environment.

In [ ]:
"""SETUP — Wajib dijalankan sekali."""
import os
import sys
import time

# Deteksi lingkungan Colab
IN_COLAB = "google.colab" in sys.modules

print(f"In Google Colab: {IN_COLAB}")
print(f"Working Dir : {os.getcwd()}")

# --- Klona repo bila diperlukan (agar impor internal berfungsi) ---
PROJECT_DIR = "collab-workspace"
if IN_COLAB:
    if PROJECT_DIR not in os.listdir():
        print("\n>>> Cloning collab-workspace...")
        os.system(
            "git clone "
            "https://github.com/febriyansyahresearch-lab/"
            f"{PROJECT_DIR}.git"
        )
    os.chdir(PROJECT_DIR)
else:
    # Fallback: naik hingga menemukan folder collab-workspace
    while True:
        cur = os.path.basename(os.getcwd())
        if cur == PROJECT_DIR or ".git" in os.listdir():
            break
        os.chdir("..")

sys.path.insert(0, os.getcwd())

print(f"\nFinal Working Dir : {os.getcwd()}")
assert os.path.exists("projects"), (
    "'projects/' tidak ditemukan. Pastikan anda berada di root collab-workspace."
)
print("[OK] Root collab-workspace terdeteksi.")

# --- Instal PyTorch bila belum ada ---
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    print("\n>>> Installing PyTorch...")
    os.system("pip install -q torch")

print("\n[DONE] Setup selesai.")

In [ ]:
"""
Import Library & Deteksi GPU
==============================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

# --- Deteksi GPU ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch   : {torch.__version__}")
print(f"Device    : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU       : TIDAK ADA → pakai CPU (aktifkan GPU di Colab: Runtime → Change runtime type → T4 GPU)")

## 📂 Load Dataset

Dataset dimuat dari file tetap `data/timeseries_fixed.csv` (365 hari data harian).

In [ ]:
"""
Load Dataset dari file CSV tetap
==================================
"""

DATA_PATH = os.path.join(os.getcwd(), "data", "timeseries_fixed.csv")
assert os.path.isfile(DATA_PATH), f"File tidak ditemukan: {DATA_PATH}"

df = pd.read_csv(DATA_PATH, parse_dates=["date"])

print(f"Dimensi dataset : {df.shape}")
print(f"Rentang tanggal : {df['date'].min().date()} s/d {df['date'].max().date()}")
print(f"Nilai min/max   : {df['value'].min():.2f} / {df['value'].max():.2f}")
print(df.head())

## 📈 Visualisasi Data

Lihat pola trend, musiman, dan noise pada data deret waktu.

In [ ]:
"""
Visualisasi Data
=================
"""

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["date"], df["value"], color="#4C72B0", linewidth=1.5)
ax.set_title("Data Deret Waktu Harian (2024)", fontsize=13, fontweight="bold")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Nilai")
plt.tight_layout()
plt.show()

# Statistik dasar
print(f"Mean   : {df['value'].mean():.2f}")
print(f"Std    : {df['value'].std():.2f}")
print(f"Trend  : naik ~{df['value'].iloc[-1] - df['value'].iloc[0]:.1f} poin dari awal ke akhir")

## 🔧 Persiapan Data untuk LSTM

Ubah deret waktu menjadi pasangan (sequence → target) dengan window `SEQ_LEN`.

In [ ]:
"""
Persiapan Data untuk LSTM
===========================
"""

SEQ_LEN = 30          # panjang window (30 hari)
BATCH_SIZE = 32
TEST_DAYS = 60        # 60 hari terakhir untuk test

values = df["value"].values.astype(np.float32)

# Normalisasi min-max
vmin, vmax = values.min(), values.max()
values_norm = (values - vmin) / (vmax - vmin)

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)

X_all, y_all = make_sequences(values_norm, SEQ_LEN)

# Split kronologis (bukan acak!)
split_idx = len(X_all) - TEST_DAYS
X_train, y_train = X_all[:split_idx], y_all[:split_idx]
X_test,  y_test  = X_all[split_idx:], y_all[split_idx:]

print(f"Total sampel     : {len(X_all)}")
print(f"Train            : {len(X_train)} ({(len(X_train)/len(X_all)*100):.0f}%)")
print(f"Test             : {len(X_test)} ({(len(X_test)/len(X_all)*100):.0f}%)")
print(f"Bentuk X_train   : {X_train.shape} (sampel, window, fitur)")

# Tensor + DataLoader
X_train_t = torch.tensor(X_train).unsqueeze(-1).to(DEVICE)
y_train_t = torch.tensor(y_train).unsqueeze(-1).to(DEVICE)
X_test_t  = torch.tensor(X_test).unsqueeze(-1).to(DEVICE)
y_test_t  = torch.tensor(y_test).unsqueeze(-1).to(DEVICE)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                         batch_size=BATCH_SIZE, shuffle=True)

## 🧠 Model LSTM

Definisikan arsitektur LSTM sederhana untuk prediksi nilai berikutnya.

In [ ]:
"""
Model LSTM
===========
"""

class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # ambil output langkah terakhir
        return out

model = LSTMPredictor().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model LSTM      : {model}")
print(f"Jumlah parameter: {n_params:,}")
print(f"Device model    : {next(model.parameters()).device}")

## ⚡ Training

Latih model dan ukur waktu per epoch. Di Colab GPU, training jauh lebih cepat.

In [ ]:
"""
Training LSTM
==============
"""

EPOCHS = 100
train_losses = []

model.train()
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(Xb)
    avg = epoch_loss / len(X_train)
    train_losses.append(avg)
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg:.6f}")

train_time = time.time() - t0
print(f"\n✅ Training selesai dalam {train_time:.2f} detik ({train_time/EPOCHS:.3f} s/epoch) di {DEVICE}")

## 📉 Kurva Loss

Visualisasikan penurunan loss selama training.

In [ ]:
"""
Kurva Loss
===========
"""

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, color="#C44E52", linewidth=1.5)
ax.set_title("Kurva Training Loss", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

print(f"Loss awal : {train_losses[0]:.6f}")
print(f"Loss akhir: {train_losses[-1]:.6f}")
print(f"Penurunan : {train_losses[0]/train_losses[-1]:.1f}x")

## 📊 Evaluasi & Prediksi

Evaluasi model pada data test dan visualisasikan prediksi vs aktual.

In [ ]:
"""
Evaluasi & Prediksi
=====================
"""

model.eval()
with torch.no_grad():
    y_pred_norm = model(X_test_t).cpu().numpy().flatten()

# Denormalisasi
y_pred = y_pred_norm * (vmax - vmin) + vmin
y_actual = y_test * (vmax - vmin) + vmin

# Metrik
rmse = np.sqrt(np.mean((y_pred - y_actual) ** 2))
mae = np.mean(np.abs(y_pred - y_actual))
mape = np.mean(np.abs((y_pred - y_actual) / y_actual)) * 100

print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"MAPE : {mape:.2f}%")

# Visualisasi prediksi vs aktual
test_dates = df["date"].iloc[-(len(y_test)):]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_dates, y_actual, label="Aktual", color="#4C72B0", linewidth=2)
ax.plot(test_dates, y_pred, label="Prediksi LSTM", color="#C44E52",
        linewidth=1.5, linestyle="--")
ax.set_title(f"Prediksi vs Aktual (RMSE={rmse:.2f})", fontsize=13, fontweight="bold")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Nilai")
ax.legend()
plt.tight_layout()
plt.show()

## ⏱️ Benchmark GPU vs CPU

Ukur kecepatan training di GPU vs CPU untuk melihat percepatannya.

In [ ]:
"""
Benchmark GPU vs CPU
=====================
"""

def benchmark(device, epochs=20):
    m = LSTMPredictor().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    m.train()
    t0 = time.time()
    for _ in range(epochs):
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(m(Xb), yb)
            loss.backward()
            opt.step()
    return (time.time() - t0) / epochs

results = {}
results["CPU"] = benchmark(torch.device("cpu"))

if torch.cuda.is_available():
    results["GPU"] = benchmark(torch.device("cuda"))
    speedup = results["CPU"] / results["GPU"]
    print(f"CPU : {results['CPU']:.4f} s/epoch")
    print(f"GPU : {results['GPU']:.4f} s/epoch")
    print(f"🚀 Percepatan GPU : {speedup:.1f}x lebih cepat")
else:
    print(f"CPU : {results['CPU']:.4f} s/epoch")
    print("⚠️  GPU tidak tersedia. Aktifkan di Colab: Runtime → Change runtime type → T4 GPU")

## 🧠 Kesimpulan

Ringkasan hasil training LSTM dengan GPU.

In [ ]:
"""
Kesimpulan
===========
"""

print("=" * 70)
print("KESIMPULAN")
print("=" * 70)
print(f"  1. Model LSTM dilatih pada {DEVICE} ({EPOCHS} epoch, {train_time:.1f} detik).")
print(f"  2. RMSE test : {rmse:.2f} | MAE : {mae:.2f} | MAPE : {mape:.2f}%")
print(f"  3. Loss turun {train_losses[0]/train_losses[-1]:.1f}x dari {train_losses[0]:.4f} → {train_losses[-1]:.4f}.")
if torch.cuda.is_available():
    print(f"  4. GPU ({torch.cuda.get_device_name(0)}) {speedup:.1f}x lebih cepat dari CPU.")
else:
    print("  4. GPU tidak tersedia saat ini — jalankan di Colab dengan runtime T4 GPU.")
print()
print("REKOMENDASI")
print("=" * 70)
print("  • Di Colab: Runtime → Change runtime type → T4 GPU, lalu jalankan ulang semua sel.")
print("  • Coba tuning: SEQ_LEN, hidden_size, num_layers, learning rate.")
print("  • Untuk data besar: gunakan batch lebih besar agar GPU lebih efisien.")
print("  • Dataset tetap (CSV) memastikan hasil reproduksibel di semua environment.")

In [ ]:
"""
Simpan Model & Push ke GitHub
===============================
"""

import subprocess

# ============================================================
# 1) TOKEN — pilih salah satu cara:
#    A) Token sudah di Secrets Colab (nama: GITHUB_TOKEN)
#    B) Atau tempel token langsung di bawah ini
# ============================================================
TOKEN = None

# Cara A: baca dari Secrets Colab
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
    print("[OK] Token dibaca dari Secrets Colab.")
except Exception:
    pass

# Cara B: tempel token langsung (hapus tanda # di depan baris ini)
# TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"   # <-- tempel token kamu di sini

assert TOKEN, "Token belum diisi! Tempel token di sel ini (lihat komentar di atas)."

# ============================================================
# 2) Simpan model LSTM
# ============================================================
MODEL_PATH = os.path.join(os.getcwd(), "projects", "timeseries", "models", "lstm_model.pt")
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "seq_len": SEQ_LEN,
    "hidden_size": 64,
    "num_layers": 2,
    "rmse": rmse,
    "mae": mae,
    "mape": mape,
    "device": str(DEVICE),
}, MODEL_PATH)

print(f"[OK] Model disimpan di: {MODEL_PATH}")
print(f"     Ukuran: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

# ============================================================
# 3) Commit & push ke GitHub
# ============================================================
REPO_URL = "https://github.com/febriyansyahresearch-lab/collab-workspace.git"
GIT_USER = "febriyansyahresearch-lab"
GIT_EMAIL = "febriyansyah.research@gmail.com"

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0:
        print(f"⚠️  {r.stderr.strip()}")
    return r.returncode

# Konfigurasi git
run(f'git config user.name "{GIT_USER}"')
run(f'git config user.email "{GIT_EMAIL}"')

# Tambah & commit
run("git add -f projects/timeseries/models/lstm_model.pt")
run(f'git commit -m "feat: LSTM model trained on {DEVICE} (RMSE={rmse:.2f})"')

# Push dengan token
push_url = REPO_URL.replace("https://", f"https://{TOKEN}@")
r = run(f"git push {push_url} main")

if r == 0:
    print("\n🎉 Berhasil di-push ke GitHub!")
else:
    print("\n⚠️  Push gagal. Cek: token benar? scope 'repo' dicentang?")